In [47]:
!pip install openai

from openai import OpenAI
from getpass import getpass

api_key = getpass("Pegá tu API Key de Groq: ")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

def preguntar(system_msg, user_msg):
    respuesta = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ]
    )
    return respuesta.choices[0].message.content

import json
import pandas as pd

print("Setup listo")

Pegá tu API Key de Groq: ··········
Setup listo


In [48]:
lista_emails = [
    """De: alfa@constructora.com | Sucursal: Palermo | Monto: $3.200.000 + IVA
    Plazo: 5 meses | Garantía: 12 meses | Inicio: marzo 2024
    Contacto: Ing. López""",

    """De: beta@obras.com | Sucursal: Belgrano | Monto: $2.800.000 IVA incluido
    Plazo: 4 meses | Garantía: 6 meses | Inicio: abril 2024
    Contacto: Arq. Fernández""",

    """De: gamma@refacciones.com | Sucursal: Palermo | Monto: $3.500.000 + IVA
    Plazo: 6 meses | Garantía: 18 meses | Inicio: febrero 2024
    Contacto: Ing. Martínez""",

    """De: delta@construcciones.com | Sucursal: Caballito | Monto: $1.900.000 IVA incluido
    Plazo: 3 meses | Garantía: 12 meses | Inicio: mayo 2024
    Contacto: Ing. Rodríguez""",

    """De: epsilon@obras.com | Sucursal: Belgrano | Monto: $4.100.000 + IVA
    Plazo: 7 meses | Garantía: 24 meses | Inicio: marzo 2024
    Contacto: Arq. Gómez"""
]

In [52]:
system = system = """Extraé del email la siguiente información y devolvela en formato JSON:

- remitente
- sucursal
- monto (respondé solo con un numero)
- iva_incluido
- plazo_meses (respondé solo con numero)
- garantia_meses (respondé solo con numero)
- inicio
- contacto

Devolvé ÚNICAMENTE el JSON válido, sin explicación ni bloques de código.
No uses formato Markdown. Los emails deben ser texto plano, por ejemplo: alfa@constructora.com"""

system_recomendacion = """Evaluá la cotización del proveedor considerando estos criterios:

- Conveniente: monto menor a $3.000.000 Y garantía mayor o igual a 12 meses
- Regular: monto entre $3.000.000 y $4.000.000 O garantía menor a 12 meses
- Poco conveniente: monto mayor a $4.000.000 O garantía menor a 6 meses

Respondé ÚNICAMENTE con una de estas tres opciones respetando exactamente el formato:
Conveniente
Regular
Poco conveniente

Sin explicación, sin puntuación, sin palabras extra."""

In [54]:
registros = []

for email in lista_emails:
    respuesta_datos = preguntar(system, email)
    respuesta_limpia = respuesta_datos.strip().replace("```json", "").replace("```", "").strip()
    datos = json.loads(respuesta_limpia)
    respuesta_recomendacion = preguntar(system_recomendacion, email)
    datos["recomendacion"] = respuesta_recomendacion.strip()
    registros.append(datos)

In [55]:
import pandas as pd
df = pd.DataFrame(registros)

In [56]:
print(df)

                  remitente   sucursal    monto  iva_incluido  plazo_meses  \
0     alfa@constructora.com    Palermo  3200000          True            5   
1            beta@obras.com   Belgrano  2800000          True            4   
2     gamma@refacciones.com    Palermo  3500000          True            6   
3  delta@construcciones.com  Caballito  1900000          True            3   
4         epsilon@obras.com   Belgrano  4100000          True            7   

   garantia_meses        inicio        contacto     recomendacion  
0              12    marzo 2024      Ing. López       Conveniente  
1               6    abril 2024  Arq. Fernández       Conveniente  
2              18  febrero 2024   Ing. Martínez           Regular  
3              12     mayo 2024  Ing. Rodríguez       Conveniente  
4              24    marzo 2024      Arq. Gómez  Poco conveniente  


In [ ]:
#PREGUNTA EJEMPLO: proveedor con el monto mas bajo
df["monto"] = pd.to_numeric(df["monto"], errors="coerce")
monto_bajo = df["monto"].idxmin()
print(df.loc[monto_bajo, ["remitente", "monto"]])

remitente    delta@construcciones.com
monto                         1900000
Name: 3, dtype: object


In [ ]:
#PREGUNTA EJEMPLO 2: proveedor que ofrece la mayor garantia
df["garantia_meses"] = pd.to_numeric(df["garantia_meses"], errors="coerce")
mayor_garantia = df["garantia_meses"].idxmax()
print(df.loc[mayor_garantia, ["remitente", "garantia_meses"]])

remitente         epsilon@obras.com
garantia_meses                   24
Name: 4, dtype: object


In [57]:
#PREGUNTA EJEMPLO 3: proveedores con IVA incluido
print(df["iva_incluido"].value_counts())

iva_incluido
True    5
Name: count, dtype: int64


In [58]:
df.to_csv("cotizaciones.csv", index=False)